# Clique — sessions B και C (framings), βήματα 9–22

**Τι κλείνει.** Μετά τα βήματα 2–8 η clique έχει τα τέσσερα κελιά του session A
(RQ2). Εδώ παίρνει και τα έξι κελιά των framings, ώστε να γίνει **πλήρης τρίτη
τοπολογία**: κάθε ερευνητικό ερώτημα, και ειδικά το **RQ3**, μετριέται σε
αστέρα, δακτύλιο και clique.

Το RQ3 χρειάζεται **και τα δύο** sessions, γιατί ο διαχωρισμός είναι ανάμεσά τους:

| session | πού μπαίνει το frame | σενάρια | σκέλη |
|---|---|---|---|
| **B** | στα **μηνύματα** (οδηγία στη φάση επικοινωνίας) | `framing_business`, `framing_team`, `framing_competitive` | cheap_talk |
| **C** | στο **system prompt** (context) | `framing_business_context`, `framing_team_context`, `framing_competitive_context` | no_comm **και** cheap_talk |

**Σύγκριση.** Με τον **αστέρα** η σύγκριση είναι καθαρή: το διορθωμένο prompt
του αστέρα είναι ταυτόσημο με το παλιό. Τα B/C του **δακτυλίου** έτρεξαν με το
παλιό prompt (το ablation κάλυψε μόνο το session A)· εκεί στηριζόμαστε στο ότι
το ablation δεν έδειξε επίδραση, και το γράφουμε ως τέτοιο.

## Τι δεν αλλάζει

PD μόνο (όπως όλη η clique), n=5, 16 γύροι, κρυφός ορίζοντας, μνήμη 10, T=0,7,
μηνύματα ως 20 λέξεις. Τα `max_tokens` τα διαλέγει το `campaign.py` ανά μοντέλο
και session, **τα ίδια με τα B/C του δακτυλίου**. (Για το gemma-2-9b αυτό
σημαίνει 192 στο B· ο αστέρας είχε 160 στο `framing_competitive` — γνωστή
διαφορά, το όριο δεν πιάνει ποτέ: ≤1/640 άκυρα.)

**Το prompt τρέχει υποχρεωτικά με `--topology-aware-comm-prompt`** — το κελί
ελέγχου παρακάτω το επιβάλλει, όπως και ότι η παράγραφος context μπαίνει στο
system prompt **και των δύο** σκελών του C.

## Setup

> **Σειρά εκτέλεσης:** πρώτα να έχει τελειώσει το session A της clique
> (`kaggle_clique.ipynb`, βήματα 2–8). Αυτό εδώ είναι τα βήματα **9–22**.

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN`, **Attach to notebook**
   *(για Llama και τα δύο gemma· τα Qwen όχι)*

Μετά: **Save Version → Save & Run All**. Αλλάζεις **μόνο** τον αριθμό `STEP`.

| βήμα | μοντέλο | τι τρέχει | runs | ~ώρες |
|---|---|---|---|---|
| 9 | gemma-2-2b | C, και τα 3 | 30 | 5,1 |
| 10 | gemma-2-2b | B, και τα 3 | 15 | 3,8 |
| 11 | Qwen2.5-7B | C, και τα 3 | 30 | 5,8 |
| 12 | Qwen2.5-7B | B, και τα 3 | 15 | 3,4 |
| 13 | Llama-3.1-8B | C: business, team | 20 | 5,1 |
| 14 | Llama-3.1-8B | C: competitive + B: competitive | 15 | 4,1 |
| 15 | Llama-3.1-8B | B: business, team | 10 | 3,1 |
| 16 | Qwen3-4B | C: business, team | 20 | 5,8 |
| 17 | Qwen3-4B | C: competitive + B: competitive | 15 | 4,9 |
| 18 | Qwen3-4B | B: business, team | 10 | 3,9 |
| 19 | gemma-2-9b | C: business | 10 | 3,7 |
| 20 | gemma-2-9b | C: team | 10 | 3,7 |
| 21 | gemma-2-9b | C: competitive + B: competitive | 15 | 6,4 |
| 22 | gemma-2-9b | B: business, team | 10 | 5,3 |

**Σύνολο 225 runs, ~64 h εκτίμηση** (~2 εβδομάδες quota). Οι ώρες βγαίνουν από
τους *μετρημένους* χρόνους της clique, με το run του `counterfactual` ως άνω
όριο για κάθε run με μήνυμα LLM· τα framings γράφουν συνήθως μικρότερα
μηνύματα, οπότε ο πραγματικός χρόνος μάλλον θα βγει μικρότερος. Κάθε βήμα
μένει κάτω από 7 ώρες.

**Προτεραιότητα αν η quota τελειώσει νωρίς:** τα βήματα με `competitive`
(το 30/30 του RQ3) πριν από τα υπόλοιπα.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU off -- Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
REPO_DIR = '/kaggle/working/repo'

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

assert os.path.exists('campaign.py'), f'campaign.py δεν είναι στο {os.getcwd()}'
for f in ('campaign.py', 'run_all_scenarios.py'):
    assert '--topology-aware-comm-prompt' in open(f, encoding='utf-8').read(), (
        f'Το {f} δεν έχει --topology-aware-comm-prompt -- κάνε git push πρώτα.')

print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'],
                              capture_output=True, text=True).stdout.strip())


## Ο έλεγχος που κρίνει το πείραμα

Χτίζει τα prompts της clique για ένα framing του C, και στα δύο σκέλη.
Επιβεβαιώνει ότι η παράγραφος context είναι μέσα, ότι το cheap-talk prompt
περιγράφει clique και όχι αστέρα, και ότι το no_comm δεν έχει καμία γραμμή
δρομολόγησης. Κοστίζει μηδέν.

In [ ]:
import sys
sys.path.insert(0, '.')
from games import GAMES
from topology import make_topology
from prompts import build_system_prompt
from message_policies import CONTEXT_FRAMING_PARAGRAPHS, get_extra_message_instruction

topo = make_topology('clique', 4)
assert topo.neighbors(0) == [1, 2, 3], topo.neighbors(0)

for frame in ('business', 'team', 'competitive'):
    ctx = CONTEXT_FRAMING_PARAGRAPHS[frame]
    common = dict(game=GAMES['pd'], n_neighbors=3, total_agents=4,
                  topology_text=topo.describe(0), context_framing_text=ctx)
    ct = build_system_prompt(condition='cheap_talk', **common,
                             communication_text=topo.describe_communication(0))
    nc = build_system_prompt(condition='no_comm', **common)
    for name, p in (('cheap_talk', ct), ('no_comm', nc)):
        assert ctx in p, f'{frame}/{name}: λείπει η παράγραφος context -- σταμάτα'
        assert 'central agent' not in p and 'peripheral' not in p, \
            f'{frame}/{name}: κείμενο αστέρα -- σταμάτα'
        assert 'fully connected' in p, f'{frame}/{name}: δεν περιγράφει clique -- σταμάτα'
    assert 'reaches every other agent' in ct, f'{frame}: λάθος δρομολόγηση -- σταμάτα'
    assert 'Messages travel' not in nc, f'{frame}: το no_comm έχει γραμμή μηνυμάτων -- σταμάτα'
    # Session B: το frame μπαίνει στη φάση μηνύματος, όχι στο system prompt.
    assert get_extra_message_instruction('framing', frame), f'{frame}: κενή οδηγία B -- σταμάτα'
    print(f'{frame:12s} C: context και στα δύο σκέλη, clique, χωρίς αστέρα   B: οδηγία μηνύματος OK')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token φορτώθηκε')
except Exception as e:
    print('Χωρίς HF_TOKEN (εντάξει μόνο για Qwen):', e)


## Ποιο βήμα

Άλλαξε **μόνο** το `STEP`. Κάθε βήμα είναι ένα μοντέλο και μία ή δύο κλήσεις
του `campaign.py` (ένα session η καθεμία). Όλα γράφουν στον ίδιο φάκελο του
μοντέλου, `<model>_clique_commfix`, δίπλα στα κελιά του session A· τα
σενάρια έχουν διαφορετικά ονόματα, οπότε δεν συγκρούονται.

In [ ]:
STEP = 9          # <-- ΜΟΝΟ ΑΥΤΟ ΑΛΛΑΖΕΙΣ (9-22)

G2B, Q25 = 'google/gemma-2-2b-it', 'Qwen/Qwen2.5-7B-Instruct'
LLA, Q3, G9B = 'meta-llama/Llama-3.1-8B-Instruct', 'Qwen/Qwen3-4B', 'google/gemma-2-9b-it'
C_ALL = ['framing_business_context', 'framing_team_context', 'framing_competitive_context']
B_ALL = ['framing_business', 'framing_team', 'framing_competitive']

# βήμα -> (μοντέλο, [(session, σενάρια), ...])
STEPS = {
    9:  (G2B, [('C', C_ALL)]),
    10: (G2B, [('B', B_ALL)]),
    11: (Q25, [('C', C_ALL)]),
    12: (Q25, [('B', B_ALL)]),
    13: (LLA, [('C', C_ALL[:2])]),
    14: (LLA, [('C', C_ALL[2:]), ('B', B_ALL[2:])]),
    15: (LLA, [('B', B_ALL[:2])]),
    16: (Q3,  [('C', C_ALL[:2])]),
    17: (Q3,  [('C', C_ALL[2:]), ('B', B_ALL[2:])]),
    18: (Q3,  [('B', B_ALL[:2])]),
    19: (G9B, [('C', C_ALL[:1])]),
    20: (G9B, [('C', C_ALL[1:2])]),
    21: (G9B, [('C', C_ALL[2:]), ('B', B_ALL[2:])]),
    22: (G9B, [('B', B_ALL[:2])]),
}
MODEL, CALLS = STEPS[STEP]
TOPOLOGY, GAMES = 'clique', ['pd']
ARMS = {s: 2 for s in C_ALL} | {s: 1 for s in B_ALL}     # C: no_comm + cheap_talk
NEEDS_HF = MODEL.startswith(('meta-llama/', 'google/'))

def args_for(session, scenarios):
    return ['--model', MODEL, '--session', session, '--topology', TOPOLOGY,
            '--scenarios', *scenarios, '--games', *GAMES,
            '--topology-aware-comm-prompt']

def expected(scenarios):
    return sum(ARMS[s] * 5 for s in scenarios)

print(f'βήμα {STEP}: {MODEL}')
for session, scenarios in CALLS:
    print(f'   session {session}: {scenarios}  -> {expected(scenarios)} runs')
print(f'   σύνολο {sum(expected(s) for _, s in CALLS)} runs')

## Preview — κοστίζει μηδέν

Για κάθε κλήση: `topology : clique`, η γραμμή `PROMPT`, το σωστό `expecting`,
φάκελος `_clique_commfix`, και `max_new_tokens` ίδιο με τα B/C του δακτυλίου.

In [ ]:
import os, subprocess, sys

ok = True
if NEEDS_HF and not os.environ.get('HUGGINGFACE_API_KEY'):
    ok = False
    print('ΛΕΙΠΕΙ ΤΟ HF_TOKEN -- το μοντέλο είναι gated, θα αποτύχει σε ~1 λεπτό.')
for session, scenarios in CALLS:
    print('=' * 70)
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(session, scenarios), '--dry-run'],
                       capture_output=True, text=True)
    out = r.stdout.strip()
    print(out[:1000])
    if r.returncode != 0:
        ok = False
        print('ΣΦΑΛΜΑ:', (r.stderr or r.stdout).strip()[-300:])
    for needle in ('topology     : clique', 'topology-aware', '_clique_commfix',
                   f'expecting    : {expected(scenarios)} run files'):
        if needle not in out:
            ok = False
            print(f'ΛΕΙΠΕΙ ΑΠΟ ΤΟ ΠΛΑΝΟ: {needle!r}')

assert ok, 'Κάποιο plan απέτυχε -- μη συνεχίσεις.'
print('\n' + '=' * 70)
print('όλα τα πλάνα εντάξει')

## Εκτέλεση

In [ ]:
import subprocess, sys, time

t0 = time.time()
results = []
for session, scenarios in CALLS:
    print('\n' + '=' * 70)
    print(f'{MODEL}  session {session}  {scenarios}   ({(time.time() - t0) / 3600:.1f} h μέχρι τώρα)')
    print('=' * 70, flush=True)
    s = time.time()
    r = subprocess.run([sys.executable, 'campaign.py', *args_for(session, scenarios)])
    mins = (time.time() - s) / 60
    status = 'OK' if r.returncode == 0 else f'FAILED (exit {r.returncode})'
    results.append((session, status, mins))
    print(f'\n--> session {session}: {status}, {mins:.0f} λεπτά', flush=True)

print('\n' + '=' * 70)
print('ΑΠΟΛΟΓΙΣΜΟΣ')
for session, s, mins in results:
    print(f'  {s:22s} {mins:6.0f} λ   session {session}')
print(f'\nσύνολο {(time.time() - t0) / 3600:.1f} h')

## Έλεγχος των δεδομένων και πρώτη ματιά

Επιβεβαιώνει clique, διορθωμένο prompt και context framing ίδιο με το όνομα
του σεναρίου, και τυπώνει τη συνεργασία ανά κελί.

In [ ]:
import glob, json, os, sys, collections, statistics
sys.path.insert(0, '.')
from analysis import summarise_run

rows = collections.defaultdict(list)
for d in sorted(glob.glob('/kaggle/working/results/*_clique_commfix')):
    for p in glob.glob(f'{d}/**/*.json', recursive=True):
        if os.path.basename(p).startswith('_') or f'{os.sep}zips{os.sep}' in p:
            continue
        rec = json.load(open(p, encoding='utf-8'))
        cfg = rec['config']
        sc = cfg.get('scenario', '')
        if not sc.startswith('framing_'):
            continue
        assert rec['topology']['type'] == 'clique', p
        assert cfg.get('topology_aware_comm_prompt') is True, f'ΛΑΘΟΣ PROMPT: {p}'
        if sc.endswith('_context'):
            assert cfg['context_framing'] == sc.split('_')[1], f'ΛΑΘΟΣ CONTEXT: {p}'
        else:
            assert cfg['message_policy'] == 'framing' and cfg['framing_type'] == sc.split('_')[1], p
        model = cfg['model']['model_id'].split('/')[-1]
        rows[(model, sc, cfg['condition'])].append(summarise_run(rec)['coop_rate_overall'])

print(f"{'model':24s} {'scenario':30s} {'arm':10s} {'n':>3s} {'coop':>6s}")
for k in sorted(rows):
    v = rows[k]
    print(f'{k[0]:24s} {k[1]:30s} {k[2]:10s} {len(v):3d} {statistics.fmean(v):6.3f}')

## Μάζεμα

In [ ]:
import glob, os, shutil

zips = sorted(glob.glob('/kaggle/working/*_clique_session[BC]_*.zip'))
print(f'{len(zips)} zip:')
for z in zips:
    print(f'   {os.path.getsize(z)/1e6:6.1f} MB  {os.path.basename(z)}')

if zips:
    box = '/kaggle/working/clique_bc'
    os.makedirs(box, exist_ok=True)
    for z in zips:
        shutil.copy(z, box)
    tag = f"step{STEP}_{MODEL.split('/')[-1]}"
    out = shutil.make_archive(f'/kaggle/working/clique_bc_{tag}', 'zip', box)
    print(f'\nκατέβασε αυτό: {out}  ({os.path.getsize(out)/1e6:.1f} MB)')
else:
    print('\nΚΑΝΕΝΑ ZIP -- δες τον απολογισμό παραπάνω.')

## Μετά

Κατέβασε το `clique_bc_step<N>_<μοντέλο>.zip`. Τα zip του μέσα αποσυμπιέζονται
στον φάκελο `<model>_clique_commfix/` του μοντέλου, δίπλα στα κελιά του A.

Η ανάγνωση, όταν τελειώσουν όλα:

1. **Το RQ3 στην clique.** Ανά μοντέλο και frame: το context `no_comm` έναντι
   του απλού `no_comm` (δρα το frame στη διάθεση;), και το B έναντι του
   `baseline` cheap talk (δρα στα μηνύματα;). Το 30/30 και το 28/30 του
   αστέρα/δακτυλίου γίνονται x/45 και y/45.
2. **Το ανταγωνιστικό frame σε δομή καρτέλ.** Η clique είναι η τοπολογία του
   Bertrand· εδώ φαίνεται πρώτη φορά αν το ανοιχτό κανάλι υπερισχύει του
   ανταγωνιστικού frame όταν μιλούν όλοι με όλους.